# Santorini Kaggle Launch Recipe

This notebook is the Kaggle version of the Santorini training recipe. Use a GPU accelerator, write checkpoints to `/kaggle/working`, and optionally copy a previous checkpoint dataset from `/kaggle/input` before resuming.

## 0. P100 PyTorch compatibility guard

Run this before importing `torch`. Kaggle's preinstalled CUDA 12.8 PyTorch wheel may not support Tesla P100 (`sm_60`), so this cell switches only P100 sessions to the official CUDA 12.6 wheel.

In [ ]:
import subprocess
import sys

try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
    ).strip()
except Exception as exc:
    gpu_name = ""
    print("Could not query GPU with nvidia-smi:", exc)

print("nvidia-smi GPU:", gpu_name or "none")
if "P100" in gpu_name:
    print("P100 detected; installing PyTorch CUDA 12.6 wheel for sm_60 support.")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "torch==2.10.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ])
    print("Install complete. If torch was imported earlier in this kernel, restart and rerun from this cell.")
else:
    print("No P100 compatibility install needed.")

## 1. Check the runtime

In [ ]:
import os
import torch

print("/kaggle/working exists:", os.path.exists("/kaggle/working"))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the repo

Enable Internet in the Kaggle notebook settings before running the clone and pip cells. If Internet is disabled, attach the repo as a Dataset instead and set `REPO_DIR` to that copied or mounted path.

In [ ]:
REPO_URL = "https://github.com/Luminous9/alpha-zero-general.git"
REPO_DIR = "/kaggle/working/alpha-zero-general"

In [ ]:
%cd /kaggle/working
!test -d "$REPO_DIR/.git" && git -C "$REPO_DIR" pull || git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

## 3. Install the Santorini training dependencies

After the runtime check succeeds, install only the lightweight Santorini dependencies. Avoid the full `requirements.txt` install unless you specifically need the older TensorFlow examples too.

In [ ]:
!pip install coloredlogs tqdm

## 4. Choose a writable checkpoint folder

In [ ]:
import os

CHECKPOINT = "/kaggle/working/Santorini-AZ/checkpoints"
os.makedirs(CHECKPOINT, exist_ok=True)
os.environ["CHECKPOINT"] = CHECKPOINT

print("Checkpoint folder:", CHECKPOINT)

## 5. Optional: import previous checkpoints

Attach a Kaggle Dataset that contains your prior checkpoint folder, then set `RESUME_CHECKPOINT_SOURCE` to that read-only `/kaggle/input/...` path. Leave it empty for a fresh run.

In [ ]:
import glob
import os
import shutil

RESUME_CHECKPOINT_SOURCE = ""  # Example: "/kaggle/input/santorini-az-checkpoints/checkpoints"

if RESUME_CHECKPOINT_SOURCE:
    if not os.path.isdir(RESUME_CHECKPOINT_SOURCE):
        raise FileNotFoundError(RESUME_CHECKPOINT_SOURCE)
    os.makedirs(CHECKPOINT, exist_ok=True)
    for path in glob.glob(os.path.join(RESUME_CHECKPOINT_SOURCE, "*")):
        destination = os.path.join(CHECKPOINT, os.path.basename(path))
        if os.path.isdir(path):
            shutil.copytree(path, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(path, destination)
    print("Copied checkpoint files from:", RESUME_CHECKPOINT_SOURCE)
else:
    print("No checkpoint dataset configured; starting from the writable checkpoint folder as-is.")

print("Checkpoint contents:")
!find "$CHECKPOINT" -maxdepth 1 -type f -printf "%f\n" | sort | tail -30

## Training seed

Run this once immediately before whichever training cell you use. The timestamp is kept for the log, and `RUN_SEED` is folded into NumPy's valid seed range.

In [ ]:
from datetime import datetime, timezone
import os

RUN_STARTED_AT = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
RUN_SEED = int(RUN_STARTED_AT) % (2**32 - 1)
os.environ["RUN_SEED"] = str(RUN_SEED)

print("Run timestamp UTC:", RUN_STARTED_AT)
print("RUN_SEED:", RUN_SEED)

## 6. First long run

If you do not already have a checkpoint, start without `--load-model`. The batch sizes below are a conservative Kaggle GPU starting point.

In [ ]:
!python main_santorini.py \
  --preset local \
  --checkpoint "$CHECKPOINT" \
  --self-play-batch-size 32 \
  --batch-size 128 \
  --arena-batch-size 32 \
  --num-iters 100 \
  --num-eps 80 \
  --num-mcts-sims 64 \
  --arena-compare 80 \
  --update-threshold 0.50 \
  --epochs 3 \
  --history-iters 5 \
  --seed "$RUN_SEED" \
  --quiet

## 7. Resume a stopped run

This loads `best.pth.tar` and replay examples from the writable checkpoint folder. If you imported prior checkpoints from `/kaggle/input`, the import cell copied them into this folder first.

In [ ]:
!python main_santorini.py \
  --preset local \
  --checkpoint "$CHECKPOINT" \
  --load-folder "$CHECKPOINT" \
  --load-model \
  --load-examples \
  --self-play-batch-size 32 \
  --batch-size 128 \
  --arena-batch-size 32 \
  --num-iters 20 \
  --num-eps 80 \
  --num-mcts-sims 64 \
  --arena-compare 80 \
  --update-threshold 0.50 \
  --epochs 3 \
  --history-iters 5 \
  --seed "$RUN_SEED" \
  --quiet

## 8. Conservative CPU or low-memory run

In [ ]:
!python main_santorini.py \
  --preset local \
  --checkpoint "$CHECKPOINT" \
  --load-folder "$CHECKPOINT" \
  --load-model \
  --load-examples \
  --self-play-batch-size 4 \
  --batch-size 64 \
  --arena-batch-size 4 \
  --num-iters 20 \
  --num-eps 80 \
  --num-mcts-sims 64 \
  --arena-compare 80 \
  --update-threshold 0.50 \
  --epochs 3 \
  --history-iters 5 \
  --seed "$RUN_SEED" \
  --quiet

Use `--examples-file filename.examples` when you want to force a specific replay file; relative paths are checked inside the load folder and from the launch directory. Use `--skip-first-self-play` only if the loaded examples were generated from the exact loaded model and you intentionally want to train immediately before collecting fresh games.

## 9. Evaluate against greedy

In [ ]:
!python pit_santorini.py \
  --baseline greedy \
  --checkpoint-folder "$CHECKPOINT" \
  --games 100 \
  --sims 64 \
  --json-out "$CHECKPOINT/eval_greedy_100_s64.json"

In [ ]:
!python pit_santorini.py \
  --baseline greedy \
  --checkpoint-folder "$CHECKPOINT" \
  --games 100 \
  --sims 128 \
  --json-out "$CHECKPOINT/eval_greedy_100_s128.json"

## 10. Save outputs for the next Kaggle run

After the notebook finishes, save or version the notebook output that contains `/kaggle/working/Santorini-AZ/checkpoints`. On the next run, attach that saved output or Dataset and point `RESUME_CHECKPOINT_SOURCE` at its checkpoint folder under `/kaggle/input`.

In [ ]:
!du -sh "$CHECKPOINT"
!find "$CHECKPOINT" -maxdepth 1 -type f -printf "%f\n" | sort | tail -50